In [2]:
!pip install -q -U google-genai sentence-transformers chromadb langchain-text-splitters pypdf

In [4]:
import os
from google.colab import userdata, files
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [6]:
api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("G_A_K")

if not api_key:
    try:
        api_key = userdata.get("Gemini_API_Key")
    except Exception:
        pass

if not api_key:
    raise ValueError("Gemini API key not found.")

os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

In [12]:
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded file:", pdf_file)

Saving GEN AI_Chatbot_87.pdf to GEN AI_Chatbot_87.pdf
Uploaded file: GEN AI_Chatbot_87.pdf


In [13]:
reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("Total characters:", len(text))
print(text[:1000])

Total characters: 1821
Disha Kanojia   Roll no 87  
 
 
NCRD’s Sterling Institute of Management Studies 
(NAAC Accredited A+ Grade) 
Nerul, Navi Mumbai 
2025-2027 
GENERATIVE AI 10-Day Workshop 
Project Documentation 
On 
Basic LLM Chatbot Using Gemini 
Under the Guidance 
of 
Mr. NAVNEET SIR 
By 
Name : Disha Manoj Kanojia 
Roll No : 87 
SYMCA Div:B 

Disha Kanojia   Roll no 87  
1. Title :- 
Basic LLM Chatbot Using Google Gemini API 
 
2. Aim :- 
To create a simple Large Language Model (LLM) application using the Google Gemini API 
in Google Colab and generate answers for user questions. 
 
3. Objective :- 
 To understand the basic concept of an LLM. 
 To connect Python with the Gemini API. 
 To use an API key securely through Google Colab Secrets. 
 To send a question to the Gemini model. 
 To display the generated answer. 
 
4. Software and Tools Used 
 
Tool Purpose 
Google Colab To write and execute Python code 
Python Programming language 
Google GenAI To communicate with G

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Number of chunks: 5

First chunk:

Disha Kanojia   Roll no 87  
 
 
NCRD’s Sterling Institute of Management Studies 
(NAAC Accredited A+ Grade) 
Nerul, Navi Mumbai 
2025-2027 
GENERATIVE AI 10-Day Workshop 
Project Documentation 
On 
Basic LLM Chatbot Using Gemini 
Under the Guidance 
of 
Mr. NAVNEET SIR 
By 
Name : Disha Manoj Kanojia 
Roll No : 87 
SYMCA Div:B


In [15]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (5, 384)


In [16]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="pdf_rag"
)

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings.tolist()
)

print("Documents stored successfully!")

Documents stored successfully!


In [17]:
def ask_question(question, top_k=3):

    # Convert question into embedding
    question_embedding = embedding_model.encode(
        [question]
    )[0].tolist()

    # Search similar chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    retrieved_chunks = results["documents"][0]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a helpful assistant.

Answer the question using only the information
provided in the context below.

If the answer is not present in the context,
say "I could not find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [ ]:
question = input("Enter your question: ")

answer = ask_question(question)

print("\nAnswer:")
print(answer)